# TempoRL / HorizonEnv — GRPO Training (Colab)

This notebook fine-tunes a small model (default: `Qwen2.5-1.5B-Instruct`) on the **HorizonEnv** inventory-management environment using **GRPO**, with the same tricks as the OpenEnv reference repo:

- DAPO asymmetric clipping (`epsilon` / `epsilon_high`)
- Dr. GRPO loss (`loss_type="dr_grpo"`)
- Truncation masking (`mask_truncated_completions=True`)

**Steps:** install deps -> clone your repo -> generate GRPO data (offline env replay) -> GRPO train with Unsloth -> quick sanity check.

> Runtime: choose **GPU (T4)** under `Runtime -> Change runtime type`.

## 1. Install dependencies

Unsloth + vLLM move fast, so if this cell errors, check https://docs.unsloth.ai for the latest Colab install snippet and swap it in — the rest of the notebook doesn't depend on exact versions.

In [ ]:
%%capture
!pip install -U pip
!pip install unsloth
!pip install --no-deps trl==0.18.0 peft accelerate bitsandbytes
!pip install fastapi pydantic python-dotenv openai

## 2. Get the repo

Replace the URL with your repo if it's different. This pulls in `models.py`, `server/`, `inference.py`, and the new `generate_grpo_data.py` / `grpo_train.py`.

In [ ]:
!git clone https://github.com/sowmyaa88/TempoRL.git
%cd TempoRL

# If generate_grpo_data.py / grpo_train.py aren't in the repo yet,
# upload them here (Files panel on the left) or copy/paste their
# contents into new files in this folder.

## 3. Generate GRPO training data

Runs a heuristic+noise policy through the env to collect ~decision points (state + replay-action history) across `easy` / `medium` / `hard` tasks. This is fast — pure Python, no GPU needed.

Start small (`--seeds-per-task 10`) to make sure everything works end-to-end, then scale up.

In [ ]:
!python generate_grpo_data.py --seeds-per-task 10 --sample-every 5 --out grpo_data.jsonl
!wc -l grpo_data.jsonl
!head -1 grpo_data.jsonl

## 4. GRPO training

Loads `Qwen2.5-1.5B-Instruct` in 4-bit with a LoRA adapter via Unsloth, then runs `GRPOTrainer` with the env-reward function defined in `grpo_train.py`.

On a free T4, keep `NUM_GENERATIONS` small (4) and watch for OOM — drop `MAX_COMPLETION_LENGTH` or `MAX_PROMPT_LENGTH` first if you hit memory limits.

In [ ]:
# Quick override knobs for a free-tier GPU — edit grpo_train.py directly
# for anything beyond these, or just run the script as-is.
import grpo_train as gt

print("Model:", gt.MODEL_NAME)
print("num_generations:", gt.NUM_GENERATIONS)
print("max_prompt_length:", gt.MAX_PROMPT_LENGTH)
print("max_completion_length:", gt.MAX_COMPLETION_LENGTH)
print("dataset size:", len(gt.dataset))

In [ ]:
gt.trainer.train()

gt.model.save_pretrained(f"{gt.OUTPUT_DIR}/final_lora")
gt.tokenizer.save_pretrained(f"{gt.OUTPUT_DIR}/final_lora")
print(f"Saved LoRA adapter to {gt.OUTPUT_DIR}/final_lora")

## 5. Sanity check: run a trained-model episode

Loads the LoRA adapter back, runs one `easy`-task episode through `InventoryEnv`, and prints the score (compare against the zero-shot `0.002` baseline from the README).

In [ ]:
from unsloth import FastLanguageModel
from grpo_train import parse_action, MODEL_NAME, MAX_SEQ_LENGTH, OUTPUT_DIR
from server.inventory_env import InventoryEnv
from server.grader import score_episode
from inference import build_prompt, SYSTEM_PROMPT

eval_model, eval_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
eval_model.load_adapter(f"{OUTPUT_DIR}/final_lora")
FastLanguageModel.for_inference(eval_model)

def get_action(obs_dict):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_prompt(obs_dict)},
    ]
    inputs = eval_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(eval_model.device)
    out = eval_model.generate(inputs, max_new_tokens=512, temperature=0.3, do_sample=True)
    text = eval_tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    action, _ = parse_action(text)
    return action

task_name = "easy"
env = InventoryEnv(task_name=task_name, seed=42)
obs = env.reset()
done = False
while not done:
    action = get_action(obs.model_dump())
    obs, reward, done = env.step(action)

print("Final profit:", env.state.total_profit)
print("Score:", score_episode(task_name, env.state.total_profit))

## Next steps

- Increase `--seeds-per-task` / decrease `--sample-every` in step 3 for more training data.
- Bump `NUM_GENERATIONS` and `NUM_TRAIN_EPOCHS` in `grpo_train.py` once a short run works end-to-end.
- Push `final_lora/` to the Hugging Face Hub (`model.push_to_hub_merged(...)` via Unsloth) to reuse it outside Colab.
- Optional: run `generate_sft_data.py` + `sft_train.py` first for an SFT warm-start, then GRPO on top — this is what the OpenEnv reference repo does.